# Notebook 02 â€” Solar EDA

Tasks:
- solar-domain EDA
- time-series EDA
- feature distributions
- outlier profiling
- relation analysis across source tables

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = "D:/data viewer/data/raw/solar_power_generation"
REPORT_DIR = "D:/data viewer/reports"

## Load Data

In [ ]:
gen1 = pd.read_csv(f"{DATA_DIR}/Plant_1_Generation_Data.csv")
gen2 = pd.read_csv(f"{DATA_DIR}/Plant_2_Generation_Data.csv")
wx1 = pd.read_csv(f"{DATA_DIR}/Plant_1_Weather_Sensor_Data.csv")
wx2 = pd.read_csv(f"{DATA_DIR}/Plant_2_Weather_Sensor_Data.csv")

def parse_datetime(series):
    for fmt in ('%d-%m-%Y %H:%M', '%Y-%m-%d %H:%M:%S'):
        try:
            return pd.to_datetime(series, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.to_datetime(series, format='mixed')

for df in [gen1, gen2, wx1, wx2]:
    df['DATE_TIME'] = parse_datetime(df['DATE_TIME'])
    df['hour'] = df['DATE_TIME'].dt.hour
    df['date'] = df['DATE_TIME'].dt.date

## Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(gen1['DC_POWER'], bins=50, alpha=0.7, label='Plant 1')
axes[0, 0].hist(gen2['DC_POWER'], bins=50, alpha=0.7, label='Plant 2')
axes[0, 0].set_title('DC Power Distribution')
axes[0, 0].legend()

axes[0, 1].hist(gen1['AC_POWER'], bins=50, alpha=0.7, label='Plant 1')
axes[0, 1].hist(gen2['AC_POWER'], bins=50, alpha=0.7, label='Plant 2')
axes[0, 1].set_title('AC Power Distribution')
axes[0, 1].legend()

axes[1, 0].hist(wx1['IRRADIATION'], bins=50, alpha=0.7, label='Plant 1')
axes[1, 0].hist(wx2['IRRADIATION'], bins=50, alpha=0.7, label='Plant 2')
axes[1, 0].set_title('Irradiation Distribution')
axes[1, 0].legend()

axes[1, 1].hist(wx1['MODULE_TEMPERATURE'], bins=50, alpha=0.7, label='Plant 1')
axes[1, 1].hist(wx2['MODULE_TEMPERATURE'], bins=50, alpha=0.7, label='Plant 2')
axes[1, 1].set_title('Module Temperature Distribution')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/eda_distributions.png', dpi=150)
plt.show()

## Hourly Patterns

In [ ]:
hourly_gen1 = gen1.groupby('hour')[['DC_POWER', 'AC_POWER']].mean()
hourly_wx1 = wx1.groupby('hour')[['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hourly_gen1.index, hourly_gen1['DC_POWER'], marker='o')
axes[0].set_title('Plant 1: Average DC Power by Hour')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('DC Power (kW)')

axes[1].plot(hourly_wx1.index, hourly_wx1['IRRADIATION'], marker='o', label='Irradiation')
axes[1].plot(hourly_wx1.index, hourly_wx1['MODULE_TEMPERATURE'], marker='s', label='Module Temp')
axes[1].set_title('Plant 1: Weather by Hour')
axes[1].set_xlabel('Hour')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/eda_hourly_patterns.png', dpi=150)
plt.show()

## Outlier Profiling

In [ ]:
for name, df, col in [('Plant 1 DC', gen1, 'DC_POWER'), ('Plant 2 DC', gen2, 'DC_POWER'),
                       ('Plant 1 Irradiation', wx1, 'IRRADIATION'), ('Plant 2 Irradiation', wx2, 'IRRADIATION')]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f"{name} {col}: {outliers} IQR outliers ({outliers/len(df)*100:.2f}%)")

## Daily Yield Analysis

In [ ]:
daily_gen1 = gen1.groupby('date')[['DC_POWER', 'AC_POWER']].mean()
daily_gen2 = gen2.groupby('date')[['DC_POWER', 'AC_POWER']].mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_gen1.index, daily_gen1['DC_POWER'], marker='o', label='Plant 1')
ax.plot(daily_gen2.index, daily_gen2['DC_POWER'], marker='s', label='Plant 2')
ax.set_title('Daily Average DC Power')
ax.set_xlabel('Date')
ax.set_ylabel('DC Power (kW)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/eda_daily_yield.png', dpi=150)
plt.show()

## Summary Statistics

In [ ]:
print("Plant 1 Generation Summary:")
print(gen1.describe())
print("\nPlant 2 Generation Summary:")
print(gen2.describe())
print("\nPlant 1 Weather Summary:")
print(wx1.describe())
print("\nPlant 2 Weather Summary:")
print(wx2.describe())

## Messy Data Story: Hospital + Flights (HoloClean)

Real dirty tables: Hospital has ~5% erroneous cells (missing, duplicates, constraint violations); Flights has multi-source conflicts (same flight, different reported times) plus heavy missingness. This is the 'berantakan → rapi' portfolio story.

In [ ]:
RAW_DIR = "D:/data viewer/data/raw"
hosp = pd.read_csv(f"{RAW_DIR}/holoclean/hospital.csv", low_memory=False)
flight = pd.read_csv(f"{RAW_DIR}/holoclean/flight.csv", low_memory=False)
print(f"hospital: {hosp.shape}, missing cells: {int(hosp.isnull().sum().sum())}")
print(f"flight: {flight.shape}, missing cells: {int(flight.isnull().sum().sum())}")
print("\nHospital missing per column (top 8):")
print(hosp.isnull().sum().sort_values(ascending=False).head(8))
print("\nFlight missing per column:")
print(flight.isnull().sum())

In [ ]:
# Denial-constraint violations in Hospital: same HospitalName must imply same ZipCode / PhoneNumber
for col in ["ZipCode", "PhoneNumber"]:
    grp = hosp.groupby("HospitalName")[col].nunique(dropna=True)
    viol = grp[grp > 1]
    print(f"HospitalName -> {col}: {len(viol)} violating hospitals (of {hosp['HospitalName'].nunique()} unique names)")
    if len(viol) > 0:
        ex = viol.index[0]
        print(hosp[hosp["HospitalName"] == ex][["HospitalName", col, "City", "State"]].head(4).to_string(index=False))
        print()

# Simple rule-based repair: fill ZipCode by mode within HospitalName, then re-check
hosp_fixed = hosp.copy()
mode_zip = hosp_fixed.groupby("HospitalName")["ZipCode"].agg(lambda s: s.mode().iloc[0] if s.notna().any() else np.nan)
hosp_fixed["ZipCode"] = hosp_fixed.apply(lambda r: mode_zip[r["HospitalName"]] if pd.isna(r["ZipCode"]) else r["ZipCode"], axis=1)
before = int((hosp.groupby("HospitalName")["ZipCode"].nunique(dropna=True) > 1).sum())
grp2 = hosp_fixed.groupby("HospitalName")["ZipCode"].nunique(dropna=True)
print(f"ZipCode violations before repair: {before}")
print(f"ZipCode violations after mode-repair: {int((grp2 > 1).sum())}")

In [ ]:
# Flights: cross-source conflicts — same flight reported differently by different sources
g = flight.groupby("flight")
conf_sched = g["scheduled_dept"].nunique(dropna=True)
conf_actual = g["actual_dept"].nunique(dropna=True)
print(f"Flights tracked: {len(g)}")
print(f"Flights with conflicting scheduled_dept: {(conf_sched > 1).sum()}")
print(f"Flights with conflicting actual_dept: {(conf_actual > 1).sum()}")
print("\nExample conflict (feeds the review queue, Problem C):")
ex = conf_sched[conf_sched > 1].index[0]
print(flight[flight["flight"] == ex][["src", "scheduled_dept", "actual_dept", "dept_gate"]].to_string(index=False))